In [6]:
!pip install mysql-connector-python

In [8]:
# Layer 1 — Raw Data Ingestion
# Pipeline: LaDe Delivery Intelligence
# Script: 01_ingest.ipynb
# Purpose: Load raw CSV into MySQL raw_deliveries table and save as Parquet for PySpark processing

import pandas as pd
import mysql.connector
from datetime import datetime
import os

print("Libraries imported successfully")

Libraries imported successfully


In [12]:
# Configuration
CONFIG = {
    "csv_file": "C:\\Users\\Rishitha\\OneDrive\\Desktop\\ds-projects\\delivery_five_cities.csv",
    "parquet_file": "raw_deliveries.parquet",
    "db_host": "localhost",
    "db_user": "root",
    "db_password": "password",
    "db_name": "lade_pipeline",
    "batch_id": datetime.now().strftime("%Y%m%d_%H%M%S")
}

print(f"Batch ID: {CONFIG['batch_id']}")
print(f"CSV file: {CONFIG['csv_file']}")

Batch ID: 20260506_114114
CSV file: C:\Users\Rishitha\OneDrive\Desktop\ds-projects\delivery_five_cities.csv


In [14]:
#  Read CSV file
if not os.path.exists(CONFIG['csv_file']):
    raise FileNotFoundError(f"CSV file not found: {CONFIG['csv_file']}")

print(f"File found. Reading {CONFIG['csv_file']}...")
start_time = datetime.now()

df_raw = pd.read_csv(CONFIG['csv_file'])

end_time = datetime.now()
duration = (end_time - start_time).seconds

print(f"File loaded successfully")
print(f"Rows: {len(df_raw)}")
print(f"Columns: {len(df_raw.columns)}")
print(f"Time taken: {duration} seconds")

File found. Reading C:\Users\Rishitha\OneDrive\Desktop\ds-projects\delivery_five_cities.csv...
File loaded successfully
Rows: 472419
Columns: 15
Time taken: 3 seconds


In [20]:
df_raw['ingested_at'] = datetime.now()
df_raw['batch_id'] = CONFIG['batch_id']
print(df_raw[['ingested_at','batch_id']].head(3))

                 ingested_at         batch_id
0 2026-05-06 11:51:21.026650  20260506_114114
1 2026-05-06 11:51:21.026650  20260506_114114
2 2026-05-06 11:51:21.026650  20260506_114114


In [24]:
#Connect to MySQL and create raw_deliveries table
conn = mysql.connector.connect(
    host=CONFIG['db_host'],
    user=CONFIG['db_user'],
    password=CONFIG['db_password'],
    database=CONFIG['db_name']
)

cursor = conn.cursor()

# Create raw_deliveries table
cursor.execute("""
    CREATE TABLE IF NOT EXISTS raw_deliveries (
        order_id VARCHAR(50),
        from_dipan_id VARCHAR(50),
        from_city_name VARCHAR(50),
        delivery_user_id VARCHAR(50),
        poi_lng VARCHAR(50),
        poi_lat VARCHAR(50),
        aoi_id VARCHAR(50),
        typecode VARCHAR(50),
        receipt_time VARCHAR(20),
        receipt_lng VARCHAR(50),
        receipt_lat VARCHAR(50),
        sign_time VARCHAR(20),
        sign_lng VARCHAR(50),
        sign_lat VARCHAR(50),
        ds VARCHAR(10),
        ingested_at DATETIME,
        batch_id VARCHAR(20)
    )
""")

conn.commit()
print("raw_deliveries table created successfully")

raw_deliveries table created successfully


In [26]:
!pip install sqlalchemy

In [30]:
# Load data into raw_deliveries table
from sqlalchemy import create_engine
import urllib.parse

# URL encode the password to handle special characters
password = urllib.parse.quote_plus(CONFIG['db_password'])

engine = create_engine(
    f"mysql+mysqlconnector://{CONFIG['db_user']}:{password}@{CONFIG['db_host']}/{CONFIG['db_name']}"
)

print("Loading data into raw_deliveries...")
load_start = datetime.now()

df_raw.to_sql(
    name='raw_deliveries',
    con=engine,
    if_exists='replace',
    index=False,
    chunksize=10000
)

load_end = datetime.now()
duration = (load_end - load_start).seconds
print(f"Data loaded successfully")
print(f"Rows loaded: {len(df_raw)}")
print(f"Time taken: {duration} seconds")

Loading data into raw_deliveries...
Data loaded successfully
Rows loaded: 472419
Time taken: 101 seconds


In [32]:
df_raw.to_parquet(CONFIG['parquet_file'], index=False)
print(f"Parquet saved: {CONFIG['parquet_file']}")

Parquet saved: raw_deliveries.parquet


In [34]:
# Log pipeline run to pipeline_runs table
end_time = datetime.now()
duration = (end_time - start_time).seconds

# Create pipeline_runs table if not exists
cursor.execute("""
    CREATE TABLE IF NOT EXISTS pipeline_runs (
        run_id INT AUTO_INCREMENT PRIMARY KEY,
        batch_id VARCHAR(20),
        layer VARCHAR(20),
        start_time DATETIME,
        end_time DATETIME,
        status VARCHAR(10),
        rows_processed INT,
        duration_seconds INT,
        error_message TEXT
    )
""")

# Insert this run's log
cursor.execute("""
    INSERT INTO pipeline_runs 
    (batch_id, layer, start_time, end_time, status, rows_processed, duration_seconds, error_message)
    VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
""", (
    CONFIG['batch_id'],
    'Layer1_Ingestion',
    start_time,
    end_time,
    'SUCCESS',
    len(df_raw),
    duration,
    None
))

conn.commit()
cursor.close()
conn.close()

print(f"Pipeline run logged successfully")
print(f"Batch ID: {CONFIG['batch_id']}")
print(f"Rows ingested: {len(df_raw)}")
print(f"Duration: {duration} seconds")
print(f"Status: SUCCESS")

Pipeline run logged successfully
Batch ID: 20260506_114114
Rows ingested: 472419
Duration: 27174 seconds
Status: SUCCESS


#### Note: duration_seconds shows time since notebook started, not just loading time.
#### In production 01_ingest.py runs as a single script — duration will be accurate.

In [38]:
# Layer 1 Summary — Raw Ingestion Complete
# Input: delivery_five_cities.csv
# Output 1: MySQL raw_deliveries table (472,419 rows)
# Output 2: raw_deliveries.parquet
# Output 3: pipeline_runs log (batch_id: 20260506_114114)
# Status: SUCCESS
print("Layer 1 complete. Moving to Layer 2 PySpark transformation.")

Layer 1 complete. Moving to Layer 2 PySpark transformation.
